In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms

In [ ]:
# Write your code here

class StandardDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # 1. Auto-generate class dictionary (or define manually)
        classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_labels = {cls_name: i for i, cls_name in enumerate(classes)}

        self.image_paths = []
        self.labels = []

        # 2. Collect Paths
        # for class_name, label in self.class_labels.items():
        #     paths = glob.glob(f"{root_dir}/{class_name}/*.*") # Matches .jpg, .png etc
        #     self.image_paths.extend(paths)
        #     self.labels.extend([label] * len(paths))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 3. Load & Process
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

transform = transforms.Compose([
    transforms.Resize((34, 34)),
    transforms.ToTensor()
])

# train_dir = os.path.join(path, "PlantVillage", "trainin")
# test_dir = os.path.join(path, "PlantVillage", "test")

train_dir = "/kaggle/input/q1-stage-3-2026/PlantVillage/test"
test_dir = "/kaggle/input/q1-stage-3-2026/PlantVillage/train"

# train_dataset = StandardDataset(root_dir=train_dir, transform=transform)
# test_dataset = StandardDataset(root_dir=test_dir, transform=transform)

train_dataset = "/kaggle/input/q1-stage-3-2026/PlantVillage/test"
test_dataset = "/kaggle/input/q1-stage-3-2026/PlantVillage/train"

# OR


from torchvision.datasets import ImageFolder

# Automatically handles everything if folders are named correctly
train_dataset = ImageFolder(root=train_dir, transform=transform)
test_dataset  = ImageFolder(root=test_dir,  transform=transform)


In [ ]:
import torch.nn as nn
import torch

# Define the CNN Model
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()

        # Convolutional Layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.batch = nn.BatchNorm1d(32 * 7 * 7)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.batch = nn.BatchNorm1d(32 * 7 * 7)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.batch = nn.BatchNorm1d(32 * 7 * 7)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=16, kernel_size=3, padding=1)
        self.batch = nn.BatchNorm1d(32 * 7 * 7)
        self.conv5 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.batch = nn.BatchNorm1d(32 * 7 * 7)
        self.conv6 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)


        self.batch = nn.BatchNorm1d(32 * 7 * 7)

        # Activation
        self.relu = nn.ReLU()

        # Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)


        # Fully Connected Layers
        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 10)  # 10 output classes for CIFAR-10

    def forward(self, x):
        # Convolution + ReLU + Pooling
        x = self.pool(self.relu(self.conv1(x)))  # (Batch, 16, 16, 16)
        x = self.pool(self.relu(self.conv2(x)))  # (Batch, 32, 8, 8)
        x = self.pool(self.relu(self.conv3(x)))  # (Batch, 64, 4, 4)
        x = self.pool(self.relu(self.conv4(x)))  # (Batch, 64, 4, 4)
        x = self.pool(self.relu(self.conv5(x)))  # (Batch, 32, 8, 8)
        x = self.pool(self.relu(self.conv6(x)))  # (Batch, 16, 16, 16)

        # Flatten
        x = x.view(x.size(0), -1)  # (Batch, 64*4*4)

        print('Right before flatten: ', x.shape)

        # x = self.gap(x)        # (Batch, 64, 1, 1)
        # x = x.flatten(1)      # (Batch, 64)


        # Fully Connected Layers
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # Logits

        return x                # Although this is a classification problem, we didn't apply softmax. Do you know why?👀 (Hint: CrossEntropyLoss has something to do here👀)
        # Because CrossEntropyLoss internally applies LogSoftmax, so the model should output raw logits to avoid applying Softmax twice.


In [ ]:
# Write your code here

from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0

    for images, _ in tqdm(dataloader):  # Ignore labels since Autoencoders don't use them
        images = images.to(device)

        _, reconstructions = model(images)  # Forward pass (encoder + decoder)
        loss = criterion(reconstructions, images)  # Compute reconstruction loss   #compare output to input

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    return avg_loss  # No accuracy since it's not classification


In [ ]:


# Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images).squeeze()  # The model outputs in shape [batch_size,1]. We convert them to [batch_size,] so the loss accepts them.
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Compute accuracy
            predictions = torch.softmax(outputs, dim=1)
            predictions = torch.argmax(predictions,dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [ ]:
import torch.optim as optim

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# Write your code here

# Write your code here
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")




In [ ]:
# Write your code here
